# LC 200 — Number of Islands
**Day 57 | Pattern: Graph Traversal (DFS Flood Fill)**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Every time you find an unvisited
'1', you've found a new island. DFS immediately floods (marks)
all connected '1's so they are never counted again. One outer
loop counts islands; DFS does the cleanup.
</div>

## Official Problem Statement

Given an `m x n` 2D binary grid `grid` which represents a map of
`'1'`s (land) and `'0'`s (water), return the number of islands.

An **island** is surrounded by water and is formed by connecting
adjacent lands horizontally or vertically. You may assume all
four edges of the grid are all surrounded by water.

**Constraints:**
- `m == grid.length`
- `n == grid[i].length`
- `1 <= m, n <= 300`
- `grid[i][j]` is `'0'` or `'1'`

## What This Is Actually Asking

Count the number of connected components in a 2-D grid where
connectivity is defined by 4-directional adjacency (no diagonals).
Each component is a group of `'1'` cells all reachable from each
other without crossing `'0'` cells.
The challenge is avoiding double-counting: once you start counting
an island you must mark all its cells so you don't count them again.
DFS is the natural tool because it exhausts a component before
returning, giving in-place marking with no extra visited structure.

## Walk Through an Example by Hand

```
Grid (4x5):           After DFS from (0,0):
1 1 1 1 0             0 0 0 0 0
1 1 0 1 0    ----->   0 0 0 0 0
1 1 0 0 0             0 0 0 0 0
0 0 0 0 0             0 0 0 0 0
```

Step-by-step outer scan:
1. `(0,0)` is `'1'` → island count = 1, DFS marks all connected
   `'1'`s as `'0'`: visits (0,0)(0,1)(0,2)(0,3)(1,0)(1,1)(1,3)
   (2,0)(2,1). Grid is now all `'0'`.
2. Outer loop continues scanning — finds no more `'1'`s.
3. Return **1**.

Second example with 3 islands:
```
1 1 0 0 0
1 1 0 0 0
0 0 1 0 0
0 0 0 1 1
```
- `(0,0)` → island 1, floods (0,0)(0,1)(1,0)(1,1)
- `(2,2)` → island 2, floods (2,2)
- `(3,3)` → island 3, floods (3,3)(3,4)
- Return **3**

## The Picture

```
Grid (3 islands):

  col: 0  1  2  3  4
row 0: 1  1  0  0  0
row 1: 1  1  0  0  0
row 2: 0  0  1  0  0
row 3: 0  0  0  1  1

DFS 4-direction offsets:
        (-1, 0)  UP
           |
(0,-1) LEFT + RIGHT (0,+1)
           |
        (+1, 0)  DOWN

Island 1 flood (from (0,0)):
  (0,0) -> right (0,1) -> down (1,0) -> down (1,1)
  All marked '0'. Stops at edges / '0' cells.

Island 2 flood (from (2,2)):
  (2,2) -> all 4 neighbours are '0', done immediately.

Island 3 flood (from (3,3)):
  (3,3) -> right (3,4) -> all neighbours '0', done.

Outer loop:
  Scans every cell left->right, top->bottom.
  Finds '1' at (0,0): count=1, DFS
  Finds '1' at (2,2): count=2, DFS
  Finds '1' at (3,3): count=3, DFS
  Return 3
```

## When To Use This Pattern

- When you need to **count connected components** in a grid,
  think DFS/BFS flood fill.
- When the problem says **"surrounded by"**, **"connected"**, or
  **"region"** in a 2-D grid, think island-style traversal.
- When modifying the grid in-place is allowed, think marking
  visited cells by overwriting (saves a separate visited set).
- When you see 4-directional or 8-directional adjacency on a
  grid, think pre-defined direction offsets + bounds check.
- When the problem is "count groups", think outer loop to
  trigger + inner DFS/BFS to exhaust each group.

## The Approach

Iterate every cell in the grid with a nested loop. Whenever you
encounter a `'1'`, increment the island counter and immediately
call DFS on that cell to flood-fill all connected land cells by
setting them to `'0'`.
The DFS function checks bounds and the current cell value; if
valid it marks the cell `'0'` then recurses in all 4 directions.
After the outer loop finishes, the counter equals the number of
distinct islands.

In [ ]:
from typing import List

In [ ]:
import copy

def test_harness(func):
    cases = [
        # (grid, expected)
        (
            [["1","1","1","1","0"],
             ["1","1","0","1","0"],
             ["1","1","0","0","0"],
             ["0","0","0","0","0"]],
            1
        ),
        (
            [["1","1","0","0","0"],
             ["1","1","0","0","0"],
             ["0","0","1","0","0"],
             ["0","0","0","1","1"]],
            3
        ),
        (
            [["0"]],
            0
        ),  # all water
        (
            [["1"]],
            1
        ),  # single land
        (
            [["1","0","1","0","1"]],
            3
        ),  # single row, alternating
        (
            [["1"],["0"],["1"],["0"],["1"]],
            3
        ),  # single col, alternating
        (
            [["1","1"],["1","1"]],
            1
        ),  # 2x2 all land
    ]
    passed = 0
    for grid, expected in cases:
        grid_copy = copy.deepcopy(grid)  # func mutates grid
        result = func(grid_copy)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status} | rows={len(grid)}x{len(grid[0])}"
                f" | expected={expected} | got={result}"
            )
    total = len(cases)
    print(f"\nResults: {passed}/{total} passed")
    if passed == total:
        print("All tests passed!")

In [ ]:
def numIslands(grid: List[List[str]]) -> int:
    """
    Count the number of islands in a binary grid.

    Strategy: DFS flood fill.
      - Outer loop finds each unvisited '1' -> new island.
      - Inner DFS marks all connected '1's as '0'.
      - 4-direction offsets: up, down, left, right.

    Args:
        grid: m x n grid of '0' (water) and '1' (land).
              NOTE: grid is modified in-place.

    Returns:
        Integer count of distinct islands.

    Time:  O(m * n)
    Space: O(m * n) worst-case recursion stack
    """
    pass
    # Debug prints (remove before submitting)
    # def dfs(r, c):
    #     print(f"  visiting ({r},{c})")
    #     ...

In [ ]:
# Uncomment and run when solution is ready
# test_harness(numIslands)

## Complexity

| Approach          | Time     | Space    | Notes                        |
|-------------------|----------|----------|------------------------------|
| Brute force       | O((mn)^2)| O(mn)    | Re-scan grid per island      |
| BFS flood fill    | O(mn)    | O(min(m,n))| Queue-based, same idea     |
| **DFS flood fill**| **O(mn)**| **O(mn)**| Stack depth = grid size      |
| Union-Find        | O(mn·α)  | O(mn)    | α ≈ 1; good for dynamic grids|

## Real World Connection

At **AWS**, detecting network subnet segmentation — finding all
reachable nodes from a VPC endpoint — is structurally identical
to counting islands in an adjacency grid.
In **data engineering**, partitioned data lake segments stored
in S3 can become "orphaned" (disconnected from the main pipeline
graph); an island-count scan finds isolated data regions.
At **Citi**, fraud cluster detection on transaction networks uses
the same connected-component logic: each fraud ring is an
"island" in a graph of suspicious account relationships.
Understanding DFS flood fill also underpins image segmentation
in ML pipelines — the exact same algorithm powers watershed
segmentation in computer vision preprocessing.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra